In [ ]:
import json
import numpy as np
from collections import Counter

file_path = '/content/drive/MyDrive/[Thesis] Improved_transformer/vivg_scenegraph_final_full_translated.json'

In [ ]:
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, dict):
    data = list(data.values())

In [ ]:
total_images = len(data)
total_regions = 0
total_relationships = 0
total_nodes = 0
total_attributes = 0
total_node_indices_in_regions = 0

len_text_en = []
len_text_vi = []

nodes_per_image = []
rels_per_image = []
regions_per_image = []

for img_data in data:
    regions = img_data.get('regions_mapping', {})
    num_regions = len(regions)
    total_regions += num_regions
    regions_per_image.append(num_regions)

    for region_info in regions.values():
        text_en = region_info.get('text_en', '')
        text_vi = region_info.get('text_vi', '')
        len_text_en.append(len(text_en.split()))
        len_text_vi.append(len(text_vi.split()))

        node_indices = region_info.get('node_indices', [])
        total_node_indices_in_regions += len(node_indices)

    scene_graph = img_data.get('global_scene_graph', {})

    nodes = scene_graph.get('nodes', [])
    num_nodes = len(nodes)
    total_nodes += num_nodes
    nodes_per_image.append(num_nodes)

    for node in nodes:
        attributes = node.get('attributes', [])
        total_attributes += len(attributes)

    relationships = scene_graph.get('relationships', [])
    num_rels = len(relationships)
    total_relationships += num_rels
    rels_per_image.append(num_rels)
    

print("\nBasic Statistics")
print(f"Total images: {total_images}")
print(f"Total regions: {total_regions}")
print(f"Avg regions/image: {total_regions/total_images:.2f}")
print(f"Max regions/image: {np.max(regions_per_image)}")
print(f"Total relationships: {total_relationships}")
print(f"Avg relationships/image: {total_relationships/total_images:.2f}")
print(f"Max relationships/image: {np.max(rels_per_image)}")

print("\nScene Graph Statistics")
print(f"Total nodes: {total_nodes}")
print(f"Avg nodes/image: {total_nodes/total_images:.2f}")
print(f"Max nodes/image: {np.max(nodes_per_image)}")
print(f"Total attributes: {total_attributes}")
print(f"Avg attributes/node: {(total_attributes/total_nodes) if total_nodes > 0 else 0:.2f}")

print("\nText & Grounding Statistics")
print(f"Avg English text length: {np.mean(len_text_en):.2f} words (Max: {np.max(len_text_en)})")
print(f"Avg Vietnamese text length: {np.mean(len_text_vi):.2f} words (Max: {np.max(len_text_vi)})")
print(f"Avg nodes referenced per region: {(total_node_indices_in_regions/total_regions) if total_regions > 0 else 0:.2f}")


Basic Statistics
Total images: 9036
Total regions: 369560
Avg regions/image: 40.90
Max regions/image: 114
Total relationships: 109744
Avg relationships/image: 12.15
Max relationships/image: 81

Scene Graph Statistics
Total nodes: 198442
Avg nodes/image: 21.96
Max nodes/image: 79
Total attributes: 259907
Avg attributes/node: 1.31

Text & Grounding Statistics
Avg English text length: 5.23 words (Max: 41)
Avg Vietnamese text length: 6.45 words (Max: 84)
Avg nodes referenced per region: 1.80


In [ ]:
canonical_counter = Counter()
attribute_counter = Counter()
alias_counts = []

total_nodes_processed = 0
nodes_without_attributes = 0

for img_data in data:
    nodes = img_data.get('global_scene_graph', {}).get('nodes', [])
    total_nodes_processed += len(nodes)

    for node in nodes:
        # Canonical
        canonical = node.get('canonical', '').lower().strip()
        if canonical:
            canonical_counter[canonical] += 1

        # Aliases
        aliases = node.get('aliases', [])
        alias_counts.append(len(aliases))

        # Attributes
        attributes = node.get('attributes', [])
        if not attributes:
            nodes_without_attributes += 1
        else:
            for attr in attributes:
                attribute_counter[attr.lower().strip()] += 1

print("Canonical & Aliases Statistics")
print(f"Unique canonicals (Object Classes): {len(canonical_counter)}")
print(f"Top 10 canonicals: {canonical_counter.most_common(10)}")
print(f"Avg aliases per node: {sum(alias_counts)/len(alias_counts):.2f}")
print(f"Max aliases for a node: {max(alias_counts)}")

print("\nAttribute Statistics")
print(f"Unique attributes: {len(attribute_counter)}")
print(f"Top 10 attributes: {attribute_counter.most_common(10)}")
print(f"Nodes without attributes: {nodes_without_attributes} ({(nodes_without_attributes/total_nodes_processed)*100:.2f}%)")

Canonical & Aliases Statistics
Unique canonicals (Object Classes): 15678
Top 10 canonicals: [('window', 3392), ('sign', 2955), ('pole', 2226), ('man', 2220), ('light', 2121), ('person', 1973), ('head', 1816), ('building', 1709), ('tree', 1536), ('wall', 1517)]
Avg aliases per node: 1.24
Max aliases for a node: 14

Attribute Statistics
Unique attributes: 8438
Top 10 attributes: [('white', 34842), ('black', 26770), ('red', 16972), ('blue', 12551), ('brown', 10759), ('wooden', 8161), ('green', 8038), ('metal', 7197), ('yellow', 5776), ('orange', 4201)]
Nodes without attributes: 78936 (39.78%)
